# 4. Feature Extraction
Bu notebook, temizlenmiş yorum verilerinden makine öğrenmesi ve derin öğrenme modelleri için özellik vektörleri (TF-IDF ve Sequence) oluşturur.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import scipy.sparse
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Veri yükleniyor...")
df = pd.read_csv('data/reviews_preprocessed.csv')
# Null değerleri temizle (önceki adımlarda kalmış olabilir)
df = df.dropna(subset=['cleaned_text'])
print(f"Veri yüklendi, Shape: {df.shape}")


c:\Users\TAHA\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Veri yükleniyor...
Veri yüklendi, Shape: (1629204, 18)


## 4.1 TF-IDF (Klasik Modeller için)

In [2]:
print("TF-IDF dönüşümü uygulanıyor...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text'])
print(f"TF-IDF Shape: {X_tfidf.shape}")

print("\nEn Sık 20 TF-IDF Özelliği:")
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
sum_tfidf = X_tfidf.sum(axis=0).A1
top_indices = sum_tfidf.argsort()[-20:][::-1]
for i, idx in enumerate(top_indices, 1):
    print(f"{i}. {feature_names[idx]} ({sum_tfidf[idx]:.2f})")


TF-IDF dönüşümü uygulanıyor...
TF-IDF Shape: (1629204, 50000)

En Sık 20 TF-IDF Özelliği:
1. food (39344.75)
2. good (34276.10)
3. place (31697.54)
4. great (27739.96)
5. service (27426.15)
6. time (24931.55)
7. like (22785.09)
8. back (21134.51)
9. get (21057.12)
10. go (20858.32)
11. one (20702.25)
12. would (19900.78)
13. order (19544.91)
14. restaurant (18681.40)
15. really (18606.76)
16. ordered (18523.50)
17. got (16840.13)
18. chicken (16595.52)
19. nice (15505.07)
20. also (15411.32)


In [3]:
print("Sayısal ekstra özellikler normalize ediliyor...")
numeric_features = [
    'review_length', 'word_count', 'exclamation_count', 'question_count',
    'avg_word_length', 'uppercase_ratio', 'sentiment_polarity', 'sentiment_subjectivity'
]

scaler = StandardScaler()
X_numeric = scaler.fit_transform(df[numeric_features])

print("TF-IDF ve Sayısal özellikler birleştiriliyor (hstack)...")
X_combined = scipy.sparse.hstack([X_tfidf, X_numeric]).tocsr()
print(f"Final Birleştirilmiş Shape: {X_combined.shape}")


Sayısal ekstra özellikler normalize ediliyor...
TF-IDF ve Sayısal özellikler birleştiriliyor (hstack)...
Final Birleştirilmiş Shape: (1629204, 50008)


In [4]:
os.makedirs('models', exist_ok=True)
joblib.dump(tfidf_vectorizer, 'models/tfidf_vectorizer.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

# TF-IDF matrisini diske kaydetmek devasa olabilir, şimdilik sadece indexleri veya 
# istendiği gibi modelleri kaydediyoruz.
print("✅ TF-IDF kaydedildi")


✅ TF-IDF kaydedildi


## 4.2 Sequence Encoding (Derin Öğrenme için)

In [5]:
print("Keras Tokenizer eğitiliyor...")
tokenizer = Tokenizer(num_words=50000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])

vocab_size = len(tokenizer.word_index) + 1
print(f"Vocab Size (Toplam Eşsiz Kelime): {vocab_size}")


Keras Tokenizer eğitiliyor...
Vocab Size (Toplam Eşsiz Kelime): 222482


In [6]:
print("Metinler dizilere (sequence) dönüştürülüyor...")
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

print("Padding işlemi uygulanıyor...")
X_seq = pad_sequences(sequences, maxlen=200, padding='post', truncating='post')
print(f"Sequence Shape: {X_seq.shape}")

print("\nÖrnek 3 Sequence:")
for i in range(3):
    print(f"Örnek {i+1}: {X_seq[i][:15]}... (uzunluk: {len(X_seq[i])})")


Metinler dizilere (sequence) dönüştürülüyor...
Padding işlemi uygulanıyor...
Sequence Shape: (1629204, 200)

Örnek 3 Sequence:
Örnek 1: [  222   216 12240    26   285  5009    42  2190   200    30    10  2634
   635   417    49]... (uzunluk: 200)
Örnek 2: [ 208  226  381   41   28   12    2 1324 2591 2029   90    7 1209  110
 1008]... (uzunluk: 200)
Örnek 3: [ 190 1073 2366  142    8  635 5752  379  353   55   10  199 1668  783
  136]... (uzunluk: 200)


In [7]:
os.makedirs('features', exist_ok=True)
joblib.dump(tokenizer, 'models/tokenizer.pkl')
np.save('features/sequences.npy', X_seq)
print("✅ Sequence kaydedildi")


✅ Sequence kaydedildi


## 4.3 Train / Validation / Test Split

In [8]:
print("Veri seti bölünüyor (%70 Train, %15 Val, %15 Test)...")
indices = np.arange(len(df))
y = df['label'].values

# İlk önce Train ve Kalan (Val+Test) olarak ayır: %70, %30
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y, test_size=0.30, random_state=42, stratify=y
)

# Kalanı Val ve Test olarak ikiye böl: %50, %50 (Her biri genel toplamın %15'i olur)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\n--- Veri Boyutları ---")
print(f"Eğitim (Train)     : {len(idx_train)}")
print(f"Doğrulama (Val)    : {len(idx_val)}")
print(f"Test (Test)        : {len(idx_test)}")

print("\n--- Sınıf Dağılımları (Stratified Doğrulaması) ---")
def print_dist(y_subset, name):
    unique, counts = np.unique(y_subset, return_counts=True)
    dist = {u: f"%{c/len(y_subset)*100:.1f}" for u, c in zip(unique, counts)}
    print(f"{name:10} Sınıf Dağılımı: {dist}")

print_dist(y_train, "Train")
print_dist(y_val, "Val")
print_dist(y_test, "Test")


Veri seti bölünüyor (%70 Train, %15 Val, %15 Test)...

--- Veri Boyutları ---
Eğitim (Train)     : 1140442
Doğrulama (Val)    : 244381
Test (Test)        : 244381

--- Sınıf Dağılımları (Stratified Doğrulaması) ---
Train      Sınıf Dağılımı: {np.int64(0): '%33.3', np.int64(1): '%33.3', np.int64(2): '%33.3'}
Val        Sınıf Dağılımı: {np.int64(0): '%33.3', np.int64(1): '%33.3', np.int64(2): '%33.3'}
Test       Sınıf Dağılımı: {np.int64(0): '%33.3', np.int64(1): '%33.3', np.int64(2): '%33.3'}


In [9]:
np.save('features/train_idx.npy', idx_train)
np.save('features/val_idx.npy', idx_val)
np.save('features/test_idx.npy', idx_test)
np.save('features/labels.npy', y)
print("✅ Split kaydedildi")

summary_df = pd.DataFrame({
    'Set': ['Train', 'Validation', 'Test'],
    'Örnek Sayısı': [len(idx_train), len(idx_val), len(idx_test)],
    'Oran (%)': [len(idx_train)/len(y)*100, len(idx_val)/len(y)*100, len(idx_test)/len(y)*100]
})
display(summary_df)


✅ Split kaydedildi


,Set,Örnek Sayısı,Oran (%)
0,Train,1140442,69.999951
1,Validation,244381,15.000025
2,Test,244381,15.000025
